## 介紹 

本課程將涵蓋： 
- 什麼是函數調用及其使用案例 
- 如何使用 Azure OpenAI 建立函數調用 
- 如何將函數調用整合至應用程式 

## 學習目標 

完成本課程後，您將知道如何並理解： 

- 使用函數調用的目的 
- 使用 Azure Open AI 服務設定函數調用 
- 為您的應用程式使用案例設計有效的函數調用 


## 了解函數呼叫

在本課程中，我們希望為我們的教育初創公司建構一個功能，讓用戶可以使用聊天機器人尋找技術課程。我們將推薦符合他們技能等級、當前職位及感興趣技術的課程。

為了完成這項任務，我們將組合使用：
 - `Azure Open AI` 來為用戶創建聊天體驗
 - `Microsoft Learn Catalog API` 來幫助用戶根據需求尋找課程
 - `函數呼叫` 將用戶的查詢傳送到函數以執行 API 請求。

首先，讓我們看看為什麼我們會想要使用函數呼叫：


### 為何要使用函數呼叫

如果你已經完成了本課程中的其他課程，你大概會了解使用大型語言模型 (LLMs) 的威力。希望你也能看到它們的一些限制。

函數呼叫是 Azure Open AI 服務的一項功能，用來克服以下限制：
1) 回應格式的一致性
2) 能夠在聊天上下文中使用應用程式其他來源的數據

在函數呼叫之前，LLM 的回應是無結構且不一致的。開發者需要撰寫複雜的驗證程式碼，以確保能處理每種不同的回應變化。

使用者無法得到像「斯德哥爾摩目前的天氣如何？」這樣的答案。這是因為模型的知識停留在訓練時的數據時間點。

讓我們看下面的例子來說明這個問題：

假設我們想建立一個學生資料庫，以便能向他們推薦適合的課程。以下是兩個學生描述，兩者所包含的資料非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想把這個資料發送給大型語言模型(LLM)去解析。稍後可以在我們的應用程式中使用此資料發送到 API 或存儲到資料庫。

我們來建立兩個相同的提示詞，指示 LLM 我們感興趣的資訊：


我們想將這個發送給大型語言模型（LLM），以解析對我們產品重要的部分。這樣我們就可以創建兩個相同的提示來指導LLM： 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


創建完這兩個提示後，我們會使用 `client.responses.create` 把它們發送給 LLM。我們將提示儲存在 `input` 變量中，並把角色設置為 `user`。這是為了模擬用戶向聊天機械人發送訊息。 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

現在我們可以將兩個請求同時發送給 LLM，並檢查我們收到的回應。


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



即使提示相同且描述相似，我們仍可能獲得不同格式的 `Grades` 屬性。

如果你執行上述儲存格多次，格式可能是 `3.7` 或 `3.7 GPA`。

這是因為大型語言模型以書寫提示的非結構化數據為輸入，並且也返回非結構化數據。我們需要有結構化格式，才能知道在儲存或使用這些數據時該期待什麼。

透過使用函數呼叫，我們可以確保收到結構化的回應。使用函數呼叫時，LLM 並不會實際呼叫或執行任何函數，而是我們為 LLM 建立一個結構讓它遵循回應。然後，我們使用這些結構化的回應來知道在我們的應用程式中應該執行哪一個函數。
 


![函數呼叫流程圖](../../../../translated_images/zh-MO/Function-Flow.083875364af4f4bb.webp)


然後我們可以將函數返回的內容傳回給 LLM。LLM 將會使用自然語言來回應，以回答用戶的查詢。


### 使用函數調用的用例 

<strong>呼叫外部工具</strong> 
聊天機器人在為使用者提供答案方面表現出色。透過使用函數調用，聊天機器人可以利用使用者的訊息來完成某些任務。例如，學生可以要求聊天機器人「發送電郵給我的導師，說我需要更多這科的協助」。這可以呼叫函數 `send_email(to: string, body: string)`


**建立 API 或資料庫查詢**
使用者可以使用自然語言查找資訊，並將其轉換成格式化的查詢或 API 請求。例如，一位老師可能會要求「誰完成了最後一次功課」，並呼叫名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函數


<strong>創建結構化資料</strong>
使用者可以利用大型語言模型從一段文字或 CSV 中提取重要資訊。例如，學生可以把一篇關於和平協議的維基百科文章轉換成 AI 閃卡。這可以透過呼叫函數 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 來完成


## 2. 建立您的第一個函式呼叫 

建立函式呼叫的過程包括 3 個主要步驟： 
1. 使用您的函式清單和用戶訊息呼叫聊天補全 API 
2. 讀取模型的回應以執行操作，例如執行函式或 API 呼叫 
3. 使用您的函式回應再次呼叫聊天補全 API，利用該資訊創建對用戶的回應。 


![函數呼叫流程](../../../../translated_images/zh-MO/LLM-Flow.3285ed8caf4796d7.webp)


### 函式呼叫的元素 

#### 使用者輸入 

第一步是建立使用者訊息。這可以透過取得文字輸入的值來動態指定，或者你也可以在這裡指定一個值。如果這是你第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。 

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（終端使用者）。對於函式呼叫，我們將其指定為 `user` 並附上一個範例問題。 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 創建函數。

接下來我們將定義一個函數以及該函數的參數。我們這裡只會使用一個名為 `search_courses` 的函數，但你可以創建多個函數。

<strong>重要</strong>：函數會包括在發送給大型語言模型的系統訊息中，並且會佔用你可用的令牌數量。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

`name` - 我們希望被調用的函數名稱。 

`description` - 這是函數如何運作的描述。這裡重要的是要具體且清晰 

`parameters` - 你希望模型在回應中產生的值和格式清單 


`type` -  屬性將被存儲的資料類型 

`properties` - 模型將用於回應的具體值清單 


`name` - 模型在格式化回應中將使用的屬性名稱 

`type` - 此屬性的資料類型 

`description` - 特定屬性的描述 


<strong>可選</strong>

`required` - 完成函數調用所需的屬性 


### 呼叫函數  
定義函數後，現在需要將它包括在 Chat Completion API 的呼叫中。我們透過在請求中添加 `functions` 來做到這一點。在此例中是 `functions=functions`。  

另外也有一個選項可以將 `function_call` 設定為 `auto`。這代表我們會讓大型語言模型根據使用者訊息決定應該呼叫哪個函數，而不是由我們自行指定。  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在我們來看看回應內容以及它的格式： 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函數的名稱被呼叫，並且根據使用者訊息，LLM 能找到符合函數參數的資料。 


## 3. 將函數調用整合到應用程式中。


在我們測試完來自 LLM 的格式化回應後，現在可以將其整合到應用程式中。

### 管理流程

要將此整合到我們的應用程式中，讓我們採取以下步驟：

首先，讓我們呼叫 Open AI 服務，並將訊息儲存在名為 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函數，用來調用 Microsoft Learn API 以獲取課程列表：


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們接著會查看模型是否想要呼叫一個函數。之後，我們會建立一個可用的函數並將其與被呼叫的函數對應起來。 
接著，我們會取得函數的參數並將它們映射到來自 LLM 的參數。

最後，我們會附加函數呼叫訊息以及由 `search_courses` 訊息返回的值。這給了 LLM 所需的所有資訊，讓它能
使用自然語言回應用戶。 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將發送更新的訊息給 LLM，以便我們能收到自然語言回應，而非 API JSON 格式的回應。


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 程式碼挑戰 

做得好！要繼續學習 Azure Open AI 函數呼叫，你可以建構: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 更多可能幫助學習者找到更多課程的函數參數。你可以在此找到可用的 API 參數： 
 - 建立另一個函數呼叫，以取得學習者更多資料，例如他們的母語 
 - 建立錯誤處理機制，當函數呼叫和/或 API 呼叫沒有回傳任何合適的課程時 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
